# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'join page', 'url': 'https://huggingface.co/join'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discourse community', 'url': 'https://discuss.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
12 days ago
•
276k
•
763
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
1 day ago
•
141k
•
1.01k
mistralai/Mistral-Small-4-119B-2603
Updated
5 days ago
•
10.3k
•
281
baidu/Qianfan-OCR
Updated
3 days ago
•
5.48k
•
278
fishaudio/s2-pro
Updated
11 days ago
•
12.3k
•
700
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.45k
Wan2.2 14B Preview
🐌
1.45k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
388
FireRed Image Edit 1.0 Fas

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n12 days ago\n•\n276k\n•\n763\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n1 day ago\n•\n141k\n•\n1.01k\nmistralai/Mistral-Small-4-119B-2603\nUpdated\n5 days ago\n•\n10.3k\n•\n281\nbaidu/Qianfan-OCR\nUpdated\n3 days ago\n•\n5.48k\n•\n278\nfishaudio/s2-pro\nUpdated\n11 days ago\n•\n12.3k\n•\n700\nBrows

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links


# Hugging Face Brochure

---

## About Hugging Face
Hugging Face is a vibrant AI community at the forefront of machine learning innovation, driven by a mission to **democratize good machine learning, one commit at a time**. The company provides a collaborative platform where developers, researchers, and enterprises come together to create, share, and explore state-of-the-art ML models, datasets, and applications across a diverse range of modalities such as text, image, video, audio, and 3D.

---

## Platform & Offerings

- **Models:** Access and collaborate on over 2 million machine learning models covering tasks like natural language processing, computer vision, and more. Popular models include Qwen3.5, Mistral Small, and various OCR and audio models.
  
- **Datasets:** Browse and contribute to over 500,000 datasets optimized for AI training and research, updated regularly by a global network of contributors.

- **Spaces:** Host and interact with machine learning applications in an easy-to-use environment that supports everything from image editing to video generation.

- **Buckets:** Manage data storage and collaboration efficiently with Hugging Face’s data hosting solutions.

- **Open Source Stack:** Hugging Face powers a fast and scalable open-source ML ecosystem enabling rapid iteration and innovation.

---

## Enterprise Solutions

Hugging Face offers tailored **Team & Enterprise Plans** designed to help organizations scale their AI capabilities with:

- **Enterprise-grade Security:** Features like Single Sign-On (SSO), identity provider integration, and regional data management.

- **Access Controls & Audit Logs:** Maintain robust governance and compliance by tracking activities and controlling access.

- **Dedicated Support:** Receive premium assistance for smooth deployments and model management.

Plans start at $20/user/month for teams, with customizable contracts available for enterprises.

---

## Company Culture

- **Community First:** Hugging Face thrives on an inclusive, open-source culture where collaboration and knowledge-sharing are key.

- **Mission-Driven:** Focused on democratizing AI and empowering developers worldwide. Every team member contributes to advancing accessible machine learning.

- **Innovation-Forward:** The team actively participates in research, publishes influential papers, and contributes to shaping future AI trends globally.

- **Learning & Growth:** With continuous updates, new learning tracks (like those on DataCamp), and a dynamic environment, Hugging Face nurtures professional development.

---

## Customers & Users

Hugging Face serves a broad spectrum of users including:

- Independent researchers and developers building AI apps.
- Enterprises needing scalable, secure AI infrastructure.
- Academic institutions and AI labs contributing to open science.
- Community members sharing datasets, models, and innovations.

The platform's extensive model and dataset library attract millions globally, fostering a vibrant, active ecosystem.

---

## Careers at Hugging Face

Join a growing team of 180+ passionate AI professionals dedicated to building the future of machine learning!  
- Work on cutting-edge AI projects and open-source tools.  
- Collaborate with a global community of contributors and experts.  
- Impact real-world AI applications across industries.  

Interested candidates can explore opportunities and contribute to the mission of democratizing AI.

---

## Stay Connected

- Website: [huggingface.co](https://huggingface.co)  
- Community: 87,000+ AI & ML practitioners actively engaging.  
- Latest research papers, articles, and blog posts available to keep you informed on AI developments.

---

## Summary

Hugging Face is more than just a platform — it’s a powerful global AI community fueling innovation through accessibility, collaboration, and open-source development. Whether you are a developer, researcher, or organization, Hugging Face offers the tools and ecosystem to accelerate your machine learning journey.

**Join Hugging Face today and be part of building the AI of tomorrow!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the leading collaboration platform for the machine learning (ML) community that is powering the future of artificial intelligence. It serves as a central hub where machine learning engineers, scientists, and end users come together to share, explore, and experiment with open-source ML models, datasets, and applications. By fostering an open and ethical AI community, Hugging Face empowers the next generation of AI developers to build smarter, more inclusive technologies.

---

## What We Offer

### The Home of Machine Learning  
- **Open collaboration platform**: Host and collaborate on unlimited public ML models, datasets, and applications.  
- **Explore AI apps**: Access millions of models and thousands of ready-to-use AI applications across text, image, video, audio, and even 3D data modalities.  
- **Build your ML portfolio**: Showcase your projects to the community and grow your professional profile.

### Key Features  
- **Models:** Browse over 2 million machine learning models ranging from NLP to computer vision and other domains.  
- **Datasets:** Work with over 500,000 datasets curated and maintained by the community and industry experts.  
- **Spaces:** Run and share ML-powered applications with an easy-to-use environment for deploying AI demos and prototypes.  
- **Enterprise Solutions:** Advanced platform features designed for teams and organizations, including security, dedicated support, audit logs, and more.

---

## Serving Our Community & Enterprise Partners

### Community Focus  
Hugging Face is home to a fast-growing, vibrant AI ecosystem where open collaboration propels innovation. Community members—from beginners to world-class AI researchers—can access resources, tools, and forums that help them accelerate their development and learning.

### Enterprise Solutions  
For organizations looking to scale AI capabilities securely and efficiently, Hugging Face offers:

- Team plans starting at $20/user/month  
- Enterprise-grade security features such as Single Sign-On (SSO), audit logs, and granular access controls  
- Private storage and dataset viewers for confidential collaboration  
- Advanced compute options including ZeroGPU quota boosts  
- Comprehensive analytics and usage tracking to optimize workflows

---

## Company Culture

Hugging Face thrives on openness, inclusivity, and innovation. The company is committed to building an ethical AI future by fostering transparency, sharing knowledge, and creating a supportive environment for learning and growth. At the heart of Hugging Face is a talented science team pushing the boundaries of AI technology and a passionate community collaborating to build the future together.

---

## Careers at Hugging Face

Join a dynamic, mission-driven company at the forefront of the AI revolution. Hugging Face hires talented individuals who are enthusiastic about open source, machine learning, and creating tools that make AI accessible to everyone. Career opportunities span AI research, software engineering, community engagement, and enterprise solutions. Work in a culture that values collaboration, continuous learning, and real-world impact.

Explore current openings and become part of shaping the future of artificial intelligence.  

---

## Connect with Hugging Face

- **Website:** https://huggingface.co  
- **Community:** GitHub, Discord, Twitter, LinkedIn  
- **Resources:** Documentation, Blog, Forum  

Build, share, and discover with Hugging Face — the AI community building the future.  

---

*Brand Colors:*  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

---

**Hugging Face** – Empowering collaboration to accelerate innovation and democratize machine learning worldwide.

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant AI community and platform dedicated to building the future of machine learning. It serves as a collaborative ecosystem where developers, researchers, and companies come together to create, discover, and share state-of-the-art machine learning models, datasets, and applications. 

The platform hosts **over 2 million models**, **500,000+ datasets**, and **1 million+ applications**, supporting multiple modalities including text, image, video, audio, and even 3D data. Hugging Face empowers users to move faster with open-source tools and accelerate their ML workflow by sharing projects and building comprehensive ML portfolios.

---

## Platform Highlights

- **Models**: Access, host, and collaborate on millions of pre-trained models covering a vast array of AI tasks such as language understanding, computer vision, speech recognition, and more.

- **Datasets**: Explore and contribute to a rich repository of datasets tailored for various machine learning domains, supporting fast experimentation and research.

- **Spaces**: Deploy and showcase ML applications easily with Hugging Face Spaces, enabling community members to create interactive AI demos and tools.

- **Open Source Stack**: Benefit from a robust and widely-adopted open-source ecosystem that accelerates innovation and integration.

- **Multimodality Support**: Work seamlessly across different data types—whether text, images, videos, audio, or 3D content.

- **Enterprise Solutions**: Tailored offerings for businesses looking to leverage AI safely and effectively with Hugging Face’s enterprise-grade tools and support.

---

## Community & Culture

Hugging Face champions an **open, collaborative, and inclusive community** culture where sharing, learning, and co-creation drive the advancement of machine learning. With a global membership base of over **87,000 followers** and active contributors, the platform fosters innovation and fast-paced progress by enabling users to:

- Host unlimited public models, datasets, and applications.
- Share their work and build their professional ML profile.
- Access the latest research papers and AI developments.
- Engage with a passionate, diverse community of AI practitioners and enthusiasts.

The team values transparency, openness, and empowerment, encouraging contributions from everyone—from hobbyists to leading AI researchers.

---

## Customers & Use Cases

Hugging Face is used by:

- **Researchers and Academics** seeking cutting-edge tools and datasets to push the boundaries of AI.
- **Developers and Data Scientists** building production-ready AI applications with pre-trained models and APIs.
- **Enterprises and Startups** deploying scalable ML solutions to enhance business processes, customer experiences, and innovation pipelines.
- **Educators and Students** leveraging accessible AI resources for learning and experimentation.

Popular trending models on the platform include large-scale language models, vision transformers, specialized audio processing networks, and more — showcasing Hugging Face as the go-to AI hub for multiple industries and research fields.

---

## Careers & Opportunities

Hugging Face is growing rapidly and often seeks passionate individuals who want to contribute to the AI revolution. Whether you are an engineer, researcher, product manager, or community specialist, joining Hugging Face means becoming part of a forward-thinking team committed to:

- Building open source AI tools that benefit the global community.
- Creating products that empower developers and enterprises worldwide.
- Fostering an inclusive and collaborative work environment.
- Driving innovation in machine learning, model hosting, and AI services.

Check Hugging Face’s website and careers page for current job openings and internship opportunities to be part of the AI community shaping tomorrow.

---

## Join the Future of AI

Explore the Hugging Face platform today to:

- Browse and contribute to 2,000,000+ models.
- Discover rich datasets for your projects.
- Create and share your own AI applications.
- Collaborate with a global AI community passionate about innovation.

**Website:** [https://huggingface.co](https://huggingface.co)

**Sign up and accelerate your machine learning journey with Hugging Face — The AI community building the future.**

---

*Empower your AI projects. Collaborate globally. Build towards tomorrow with Hugging Face.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>